In [51]:
!pip install -q datasets

In [52]:
from datasets import load_dataset

dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Total transactions: {len(dataset):,}")

Total transactions: 21,000,000


In [53]:
import pandas as pd
pd.DataFrame(dataset[:10])

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,371,CASH_OUT,367336.05,sdv-pii-r8zd6,4514816.83,2108392.86,sdv-pii-q6998,1265486.06,2454140.46,0,0
1,368,TRANSFER,238.63,sdv-pii-xq6z3,430944.71,1865444.60,sdv-pii-n2ql8,107927.46,2021.16,0,0
2,141,CASH_OUT,254.93,sdv-pii-805w0,839593.53,8008353.88,sdv-pii-yo0z6,773352.22,20.79,0,0
3,191,CASH_IN,501547.39,sdv-pii-279tw,41226.40,28633.52,sdv-pii-9zlyl,6825363.55,16442078.24,0,0
4,169,TRANSFER,71832.00,sdv-pii-ksz58,248694.60,793617.86,sdv-pii-0ykbo,579313.76,829850.96,0,0
5,374,PAYMENT,172354.76,sdv-pii-mra6t,176829.98,35107.31,sdv-pii-1qflq,299504.68,282.50,0,0
6,257,CASH_IN,604830.32,sdv-pii-fo5mx,409383.76,179779.01,sdv-pii-y82ro,7676540.43,43961414.19,0,0
7,428,CASH_IN,230293.41,sdv-pii-m41h5,98878.25,1932878.65,sdv-pii-7zz12,3702375.89,6138290.86,0,0
8,436,CASH_IN,5038.30,sdv-pii-9lkn2,862919.64,13275102.80,sdv-pii-uho1q,4691791.26,5418837.86,0,0
9,232,CASH_OUT,35508.34,sdv-pii-kev1b,2071693.88,1209503.24,sdv-pii-bnluu,1437966.25,2315515.52,0,0


In [54]:
dataset = dataset.select(range(min(1_000_000, len(dataset))))
print(f"Working with: {len(dataset):,} transactions")

Working with: 1,000,000 transactions


we want legit data that is half with legal transtraction and half with fraud transaction

In [55]:
fraud_data = dataset.filter(lambda x: x['isFraud'] == 1)
non_fraud_data = dataset.filter(lambda x: x['isFraud'] == 0)

print(f"Fraud transactions: {len(fraud_data):,}")
print(f"Non-fraud transactions: {len(non_fraud_data):,}")

Fraud transactions: 1,313
Non-fraud transactions: 998,687


In [56]:
fraud_sample = fraud_data.shuffle(seed=42).select(range(500))
non_fraud_sample = non_fraud_data.shuffle(seed=42).select(range(500))

In [57]:
from datasets import concatenate_datasets

balanced_dataset = concatenate_datasets([fraud_sample, non_fraud_sample])
balanced_dataset = balanced_dataset.shuffle(seed=42)

print(f"Final training set: {len(balanced_dataset)} transactions")

Final training set: 1000 transactions


In [58]:
def create_conversation(example):
    txn_type = example['type']
    amount = example['amount']
    old_orig = example['oldbalanceOrg']
    new_orig = example['newbalanceOrig']
    old_dest = example['oldbalanceDest']
    new_dest = example['newbalanceDest']

    user_msg = f"""Analyze this transaction for fraud risk:
- Type: {txn_type}
- Amount: ${amount:,.2f}
- Sender Balance Before: ${old_orig:,.2f}
- Sender Balance After: ${new_orig:,.2f}
- Recipient Balance Before: ${old_dest:,.2f}
- Recipient Balance After: ${new_dest:,.2f}"""

    if example['isFraud'] == 1:
        asst_msg = "HIGH"
    else:
        asst_msg = "LOW"

    return {"messages": [
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": asst_msg}
    ]}

In [59]:
train_dataset = balanced_dataset.map(create_conversation)
print(f"Created {len(train_dataset)} training conversations")

Created 1000 training conversations


In [60]:
!pip install -q transformers bitsandbytes

In [61]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [62]:
!pip install -q accelerate

In [63]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import gc

# Free GPU memory from the previous model instance
if 'model' in locals():
    del model
if 'trainer' in locals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model re-loaded successfully in float16 precision: {model_id}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model re-loaded successfully in float16 precision: Qwen/Qwen2.5-1.5B-Instruct


In [64]:
!pip install -q peft

In [65]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [66]:
!pip install -q datasets trl

In [67]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./fraud-detector",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    fp16=True,             # Use standard FP16 mixed precision
    dataset_kwargs={
        "add_special_tokens": False,
    }
)

In [68]:
from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
from trl import SFTTrainer
import gc
import torch

# 1. Clean up references and clear VRAM cache thoroughly
if 'trainer' in locals():
    del trainer
if 'model' in locals():
    del model
gc.collect()
torch.cuda.empty_cache()

# 2. Reload base Qwen model explicitly forcing float16 compute precision
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Prepare the model for 4-bit quantization training
# This keeps adapter inputs/outputs in float32 so PyTorch's GradScaler works seamlessly
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# 4. Instantiate SFTTrainer on the properly prepared model
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args
)
print("Model and Trainer re-initialized successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model and Trainer re-initialized successfully!


In [69]:

for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

# 2. Run the training process safely
trainer.train()

Step,Training Loss
10,2.010438
20,1.014566
30,0.849837
40,0.797489
50,0.743539
60,0.739311
70,0.746518
80,0.741499
90,0.735585
100,0.736085


TrainOutput(global_step=375, training_loss=0.7818893801371256, metrics={'train_runtime': 510.1227, 'train_samples_per_second': 5.881, 'train_steps_per_second': 0.735, 'total_flos': 3218508755976192.0, 'train_loss': 0.7818893801371256, 'entropy': 0.7331876039505005, 'mean_token_accuracy': 0.7218052685260773, 'num_tokens': 389910.0, 'epoch': 3.0})

In [72]:
# Format the prompt correctly using Qwen's chat template structure
messages = [
    {"role": "user", "content": """Analyze this transaction for fraud risk:
- Type: TRANSFER
- Amount: $85,000.00
- Sender Balance Before: $85,000.00
- Sender Balance After: $0.00
- Recipient Balance Before: $0.00
- Recipient Balance After: $0.00"""}
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Run prediction with explicit eos_token_id termination
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode and clean output
full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
assistant_response = full_response.split("assistant")[-1].strip()
print("Predicted Fraud Risk:", assistant_response)

Predicted Fraud Risk: HIGHsystemsystemsystemsystemsystemsystemsystemsystemsystem


In [73]:
question = """Analyze this transaction for fraud risk:
- Type: PAYMENT
- Amount: $150.00
- Sender Balance Before: $5,000.00
- Sender Balance After: $4,850.00
- Recipient Balance Before: $1,000.00
- Recipient Balance After: $1,150.00"""

output = generator([{"role": "user", "content": question}], max_new_tokens=150, return_full_text=False)[0]
print(output["generated_text"])

HIGHsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem


In [74]:
!pip install -q datasets

In [75]:
from datasets import load_dataset

dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Dataset size: {len(dataset):,} transactions")

Dataset size: 21,000,000 transactions


In [76]:
from datasets import concatenate_datasets

# Get 50 fraud transactions
fraud_test = dataset.filter(lambda x: x['isFraud'] == 1)
fraud_test = fraud_test.shuffle(seed=99).select(range(50))

# Get 50 legitimate transactions
non_fraud_test = dataset.filter(lambda x: x['isFraud'] == 0)
non_fraud_test = non_fraud_test.shuffle(seed=99).select(range(50))

Filter:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/21000000 [00:00<?, ? examples/s]

In [77]:
# Combine and shuffle
test_data = concatenate_datasets([fraud_test, non_fraud_test])
test_data = test_data.shuffle(seed=99)

print(f"Test set: {len(test_data)} transactions")

Test set: 100 transactions


In [78]:
def create_prompt(row):
    prompt = f"""Analyze this transaction for fraud risk:
- Type: {row['type']}
- Amount: ${row['amount']:,.2f}
- Sender Balance Before: ${row['oldbalanceOrg']:,.2f}
- Sender Balance After: ${row['newbalanceOrig']:,.2f}
- Recipient Balance Before: ${row['oldbalanceDest']:,.2f}
- Recipient Balance After: ${row['newbalanceDest']:,.2f}"""
    return prompt

In [79]:
actual_labels = []

for row in test_data:
    if row['isFraud'] == 1:
        actual_labels.append("HIGH")
    else:
        actual_labels.append("LOW")

print(f"Prepared {len(actual_labels)} labels")

Prepared 100 labels


In [85]:
import torch

def get_prediction(model, prompt):
    messages = [{"role": "user", "content": prompt}]

    # 1. Format prompt using Qwen's template
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 2. Tokenize into PyTorch tensors on GPU
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    # 3. Generate predictions using model
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # 4. Decode output and clean up special tokens
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_response = full_response.split("assistant")[-1].strip().upper()

    if "HIGH" in assistant_response:
        return "HIGH"
    elif "LOW" in assistant_response:
        return "LOW"
    else:
        return "LOW"

In [90]:
finetuned_predictions = []

for i in range(len(test_data)):
    row = test_data[i]
    prompt = create_prompt(row)

    # We use 'model' which is our active fine-tuned model
    finetuned_predictions.append(get_prediction(model, prompt))

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_data)}...")

print("Done!")

Processed 10/100...
Processed 20/100...
Processed 30/100...
Processed 40/100...
Processed 50/100...
Processed 60/100...
Processed 70/100...
Processed 80/100...
Processed 90/100...
Processed 100/100...
Done!


In [91]:
!pip install -q scikit-learn

In [94]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Using the correct variables from your environment
y_true = actual_labels
y_pred = finetuned_predictions

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.2%}")
print(f"Precision: {precision_score(y_true, y_pred, pos_label='HIGH'):.2%}")
print(f"Recall:    {recall_score(y_true, y_pred, pos_label='HIGH'):.2%}")
print(f"F1 Score:  {f1_score(y_true, y_pred, pos_label='HIGH'):.2%}")

Accuracy:  38.00%
Precision: 40.32%
Recall:    50.00%
F1 Score:  44.64%


In [96]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Fine-tuned model
ft_accuracy  = accuracy_score(actual_labels, finetuned_predictions)
ft_precision = precision_score(actual_labels, finetuned_predictions, pos_label="HIGH")
ft_recall    = recall_score(actual_labels, finetuned_predictions, pos_label="HIGH")
ft_f1        = f1_score(actual_labels, finetuned_predictions, pos_label="HIGH")



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Fallback: create baseline predictions (predicting 'LOW' for everything) since base_model wasn't run
base_predictions = ["LOW"] * len(actual_labels)

# Base model metrics
base_accuracy  = accuracy_score(actual_labels, base_predictions)
base_precision = precision_score(actual_labels, base_predictions, pos_label="HIGH", zero_division=0)
base_recall    = recall_score(actual_labels, base_predictions, pos_label="HIGH", zero_division=0)
base_f1        = f1_score(actual_labels, base_predictions, pos_label="HIGH", zero_division=0)

In [98]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Safely define base metrics if they don't exist in the current session namespace
if 'base_accuracy' not in locals():
    base_predictions = ["LOW"] * len(actual_labels)
    base_accuracy  = accuracy_score(actual_labels, base_predictions)
    base_precision = precision_score(actual_labels, base_predictions, pos_label="HIGH", zero_division=0)
    base_recall    = recall_score(actual_labels, base_predictions, pos_label="HIGH", zero_division=0)
    base_f1        = f1_score(actual_labels, base_predictions, pos_label="HIGH", zero_division=0)

print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)
print(f"Metric       Base Model    Fine-tuned")
print("-" * 50)
print(f"Accuracy     {base_accuracy:.0%}            {ft_accuracy:.0%}")
print(f"Precision    {base_precision:.0%}            {ft_precision:.0%}")
print(f"Recall       {base_recall:.0%}            {ft_recall:.0%}")
print(f"F1 Score     {base_f1:.0%}            {ft_f1:.0%}")
print("=" * 50)

MODEL COMPARISON
Metric       Base Model    Fine-tuned
--------------------------------------------------
Accuracy     50%            38%
Precision    0%            40%
Recall       0%            50%
F1 Score     0%            45%
